# Notebook 01 — ETL Pipeline — Student Version

## Goal

Build a **versioned, privacy-aware ML feature dataset** from the QBC12 Airbnb PostgreSQL database.

Final output:

- one row per `listing_id`
- one fixed `cutoff_date`
- features built only from data available before/on the cutoff
- target built from future calendar availability
- no raw PII columns in the final ML dataset

The next notebook will use this output for MLflow experiments. If this ETL is messy, the ML notebook will be garbage.

## What you must submit from this notebook

By the end, your notebook must save these files under `data/features/`:

```text
listing_availability_features_<version>.csv
listing_availability_features_<version>.parquet
listing_availability_features_<version>_metadata.json
listing_availability_features_<version>_validation_report.json
pii_audit_<version>.csv
```

The notebook must also show:

1. database connection check,
2. table/column inspection,
3. PII audit,
4. cutoff-date logic,
5. feature construction,
6. label construction,
7. validation checks.

## 0. Imports

These libraries are enough for the ETL notebook.

Install missing packages with:

```bash
pip install pandas numpy sqlalchemy psycopg2-binary pyarrow
```

In [1]:
import os
import json
import re
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## 1. Configuration

These values define the dataset version and the time windows.

- `PAST_WINDOW_DAYS`: how much history is used for features.
- `FUTURE_WINDOW_DAYS`: how much future data is used for the target.
- `HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD`: the rule for the positive class.

If you change any of these, change `DATASET_VERSION`.

In [2]:
# -----------------------------
# ETL Configuration
# -----------------------------
DATASET_VERSION = "v1_student"

ENTITY_COLUMN = "listing_id"

PAST_WINDOW_DAYS = 90
FUTURE_WINDOW_DAYS = 30
HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD = 0.30

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FEATURE_DIR = DATA_DIR / "features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_DIR:", FEATURE_DIR)

PROJECT_ROOT: /Users/parhamrahimi/Documents/bluBank/Quera/HW2/A
FEATURE_DIR: /Users/parhamrahimi/Documents/bluBank/Quera/HW2/A/data/features


## 2. Database connection

Use your assigned student database user.

The QBC12 database is:

host: 185.50.38.163

port: 32112

database: qbc12_airbnb

Important:

- Keep `sslmode=disable`.
- Do not commit real passwords to Git.

In [3]:
# -----------------------------
# Database Connection
# -----------------------------

# Clear old environment variables that may point to the wrong database.
# for key in ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]:
#     os.environ.pop(key, None)

DB_HOST = os.getenv("PGHOST", "185.50.38.163")
DB_PORT = int(os.getenv("PGPORT", "32112"))
DB_NAME = os.getenv("PGDATABASE", "qbc12_airbnb")
DB_USER = os.getenv("PGUSER", "")
DB_PASSWORD = os.getenv("PGPASSWORD", "")


if DB_USER == "student_your_username" or DB_PASSWORD == "student_your_password":
    raise ValueError("Replace DB_USER and DB_PASSWORD with your assigned database credentials.")

print("Connecting to:")
print("HOST:", DB_HOST)
print("PORT:", DB_PORT)
print("DB:", DB_NAME)
print("USER:", DB_USER)

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    query={"sslmode": "disable"},
)

engine = create_engine(db_url)

def read_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query through SQLAlchemy and return a Pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

with engine.connect() as conn:
    connection_check = conn.execute(
        text("""
        SELECT
            current_database() AS database,
            current_user AS user_name,
            inet_server_addr() AS server_ip,
            inet_server_port() AS server_port,
            now() AS checked_at;
        """)
    ).mappings().first()

dict(connection_check)

Connecting to:
HOST: 185.50.38.163
PORT: 32112
DB: qbc12_airbnb
USER: student_nazanin_hesari


OperationalError: (psycopg2.OperationalError) connection to server at "185.50.38.163", port 32112 failed: FATAL:  remaining connection slots are reserved for roles with the SUPERUSER attribute

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 3. Inspect the available data

Before writing ETL, inspect the database.

You should confirm:

- which tables exist,
- which columns exist,
- how many rows each table has,
- whether important fields are missing.

In [ ]:
tables_df = read_sql("""
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
""")

tables_df

In [ ]:
columns_df = read_sql("""
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'core'
ORDER BY table_schema, table_name, ordinal_position;
""")

columns_df

In [ ]:
row_counts_df = read_sql("""
SELECT 'core.calendar_day' AS table_name, COUNT(*) AS row_count FROM core.calendar_day
UNION ALL
SELECT 'core.host' AS table_name, COUNT(*) AS row_count FROM core.host
UNION ALL
SELECT 'core.listing' AS table_name, COUNT(*) AS row_count FROM core.listing
UNION ALL
SELECT 'core.neighbourhood' AS table_name, COUNT(*) AS row_count FROM core.neighbourhood
UNION ALL
SELECT 'core.review' AS table_name, COUNT(*) AS row_count FROM core.review
ORDER BY table_name;
""")

row_counts_df

## 4. Data quality audit

This step decides which columns are safe and useful.

You must check at least:

1. calendar date range,
2. review date range,
3. whether `calendar_day.price` and `adjusted_price` are usable,
4. whether recent review windows are meaningful.

Do not include columns that are all-null or nearly useless.

In [ ]:
calendar_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE price IS NULL) AS null_price,
    COUNT(*) FILTER (WHERE adjusted_price IS NULL) AS null_adjusted_price,
    COUNT(*) FILTER (WHERE available IS NULL) AS null_available,
    MIN(date) AS min_calendar_date,
    MAX(date) AS max_calendar_date
FROM core.calendar_day;
""")

review_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE comment_len IS NULL) AS null_comment_len,
    MIN(review_date) AS min_review_date,
    MAX(review_date) AS max_review_date
FROM core.review;
""")

display(calendar_quality_df)
display(review_quality_df)

In [ ]:
# Inspect small samples.
# Keep LIMIT small. Do not pull full raw calendar/review tables into Pandas.

for table_name in ["listing", "host", "neighbourhood", "review", "calendar_day"]:
    print(f"\n===== core.{table_name} =====")
    display(read_sql(f"SELECT * FROM core.{table_name} LIMIT 10;"))

## 5. Choose the cutoff date

The cutoff separates features from the label.

Rules:

- Historical features use dates `history_start_date <= date <= cutoff_date`.
- The label uses dates `cutoff_date < date <= label_end_date`.
- The cutoff must have enough past calendar data and enough future calendar data.

For this homework, use a 90-day feature window and a 30-day label window.

In [ ]:
range_df = read_sql("""
SELECT
    (SELECT MIN(date) FROM core.calendar_day) AS calendar_min_date,
    (SELECT MAX(date) FROM core.calendar_day) AS calendar_max_date,
    (SELECT MIN(review_date) FROM core.review) AS review_min_date,
    (SELECT MAX(review_date) FROM core.review) AS review_max_date;
""")

range_df

In [ ]:
calendar_min_date = pd.to_datetime(range_df.loc[0, "calendar_min_date"]).date()
calendar_max_date = pd.to_datetime(range_df.loc[0, "calendar_max_date"]).date()

# Compute earliest valid cutoff: calendar_min_date + PAST_WINDOW_DAYS
# Compute latest valid cutoff: calendar_max_date - FUTURE_WINDOW_DAYS

earliest_cutoff_allowed_by_calendar = calendar_min_date + timedelta(days=PAST_WINDOW_DAYS)
latest_cutoff_allowed_by_calendar = calendar_max_date - timedelta(days=FUTURE_WINDOW_DAYS)

# Choose a valid cutoff_date: use the latest valid cutoff to maximize data
cutoff_date = latest_cutoff_allowed_by_calendar

# Compute history_start_date and label_end_date
history_start_date = cutoff_date - timedelta(days=PAST_WINDOW_DAYS)
label_end_date = cutoff_date + timedelta(days=FUTURE_WINDOW_DAYS)

print("Calendar min date:", calendar_min_date)
print("Calendar max date:", calendar_max_date)
print("Earliest cutoff allowed:", earliest_cutoff_allowed_by_calendar)
print("Latest cutoff allowed:", latest_cutoff_allowed_by_calendar)
print()
print("Chosen cutoff_date:", cutoff_date)
print("History start date:", history_start_date)
print("Label end date:", label_end_date)

# Validate cutoff is within allowed range
assert earliest_cutoff_allowed_by_calendar <= cutoff_date <= latest_cutoff_allowed_by_calendar, \
    "cutoff_date must be between earliest and latest allowed cutoffs"
assert history_start_date >= calendar_min_date, \
    "history_start_date must be >= calendar_min_date"
assert label_end_date <= calendar_max_date, \
    "label_end_date must be <= calendar_max_date"

print()
print("All date assertions passed.")

## 6. PII audit

Raw identifiers can be needed for joins, but they must not become model features.

Your final ML feature table must not contain:

- `host_id`
- `host_pseudo_id`
- `review_id`
- `reviewer_id`
- `reviewer_pseudo_id`
- `license`
- raw text fields that may contain sensitive information

`listing_id` may stay as an entity key, but it must be excluded from model inputs later.

In [ ]:
pii_audit = pd.DataFrame([
    {
        "table": "listing",
        "column": "listing_id",
        "pii_type": "entity identifier",
        "decision": "keep as entity key only",
        "reason": "needed to define one row per listing; not a model input"
    },
    {
        "table": "listing",
        "column": "host_id",
        "pii_type": "entity identifier",
        "decision": "drop from final features",
        "reason": "used only for joins to host table; must not become a model feature"
    },
    {
        "table": "host",
        "column": "host_pseudo_id",
        "pii_type": "pseudo-identifier",
        "decision": "drop from final features",
        "reason": "pseudo-identifier that could be used to de-anonymize hosts"
    },
    {
        "table": "listing",
        "column": "license",
        "pii_type": "license/registration number",
        "decision": "drop from final features",
        "reason": "could contain identifiable registration or license information"
    },
    {
        "table": "review",
        "column": "review_id",
        "pii_type": "entity identifier",
        "decision": "never loaded into Pandas (aggregated in SQL)",
        "reason": "only used for SQL aggregation; not needed in final dataset"
    },
    {
        "table": "review",
        "column": "reviewer_id",
        "pii_type": "entity identifier",
        "decision": "never loaded into Pandas (aggregated in SQL)",
        "reason": "only used for counting unique reviewers in SQL; not a model feature"
    },
    {
        "table": "review",
        "column": "reviewer_pseudo_id",
        "pii_type": "pseudo-identifier",
        "decision": "never loaded into Pandas (aggregated in SQL)",
        "reason": "pseudo-identifier; only needed in SQL for unique reviewer counts"
    },
    {
        "table": "listing",
        "column": "bathrooms_text",
        "pii_type": "raw text field",
        "decision": "drop after parsing into numeric bathrooms",
        "reason": "raw text field that may contain unexpected information; keep only parsed numeric value"
    },
])

pii_audit

## 7. Extract static tables

`listing`, `host`, and `neighbourhood` are small enough to load directly.

Do not load full `review` or `calendar_day` into Pandas. Those must be aggregated in SQL later.

In [ ]:
# Load required listing columns.
# Exclude: license (PII), bathrooms_text will be parsed and dropped later.
# Keep: listing_id, host_id (for join only), neighbourhood_id, key property fields.
listing_df = read_sql("""
SELECT
    listing_id,
    host_id,
    neighbourhood_id,
    property_type,
    room_type,
    accommodates,
    bathrooms_text,
    bedrooms,
    beds,
    listing_price,
    minimum_nights,
    maximum_nights,
    instant_bookable
FROM core.listing;
""")

# Load host columns.
# Exclude: host_pseudo_id (PII)
# Keep: host_id, is_superhost
host_df = read_sql("""
SELECT
    host_id,
    is_superhost
FROM core.host;
""")

# Load neighbourhood columns and rename name to neighbourhood_name.
neighbourhood_df = read_sql("""
SELECT
    neighbourhood_id,
    name AS neighbourhood_name
FROM core.neighbourhood;
""")

print("listing:", listing_df.shape)
print("host:", host_df.shape)
print("neighbourhood:", neighbourhood_df.shape)

## 8. Clean static fields

Convert database values into ML-friendly columns.

Required work:

- convert booleans to boolean dtype,
- convert numeric listing columns to numeric dtype,
- parse `bathrooms_text` into a numeric `bathrooms` feature.

In [ ]:
# Normalize boolean columns.
listing_df["instant_bookable"] = listing_df["instant_bookable"].map(
    {"t": True, "f": False, True: True, False: False}
).astype(bool)

host_df["is_superhost"] = host_df["is_superhost"].map(
    {"t": True, "f": False, True: True, False: False}
).astype(bool)

# Normalize numeric listing columns.
numeric_cols_listing = [
    "accommodates",
    "bedrooms",
    "beds",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
]

for col in numeric_cols_listing:
    listing_df[col] = pd.to_numeric(listing_df[col], errors="coerce")


def parse_bathrooms(text_value):
    """
    Convert bathrooms_text into a number.

    Examples:
    - '1 bath' -> 1.0
    - '1.5 baths' -> 1.5
    - 'Half-bath' -> 0.5
    - missing/unrecognized -> NaN
    """
    if pd.isna(text_value) or not isinstance(text_value, str):
        return np.nan

    text_lower = text_value.strip().lower()

    # Handle 'half-bath' explicitly
    if "half" in text_lower:
        return 0.5

    # Try to extract a number from the text (e.g., '1 bath', '1.5 baths')
    match = re.search(r'(\d+\.?\d*)', text_lower)
    if match:
        return float(match.group(1))

    return np.nan


listing_df["bathrooms"] = listing_df["bathrooms_text"].apply(parse_bathrooms)

listing_df[["bathrooms_text", "bathrooms"]].head(10)

## 9. Build static listing features

Join:

- `listing` → `host`
- `listing` → `neighbourhood`
- host-level aggregate `host_listing_count`

Final static features should be one row per `listing_id`.

Do not keep raw `host_id`, `host_pseudo_id`, `neighbourhood_id`, `license`, or `bathrooms_text` in the final static feature table.

In [ ]:
# Create host_listing_features with one row per host_id.
# Count how many listings each host has.
host_listing_features = (
    listing_df
    .groupby("host_id")
    .agg(host_listing_count=("listing_id", "nunique"))
    .reset_index()
)

# Merge listing with host.
base_listing_features = listing_df.merge(host_df, on="host_id", how="left")

# Merge with host_listing_features.
base_listing_features = base_listing_features.merge(
    host_listing_features, on="host_id", how="left"
)

# Merge with neighbourhood.
base_listing_features = base_listing_features.merge(
    neighbourhood_df, on="neighbourhood_id", how="left"
)

# Choose privacy-safe static feature columns.
# Exclude: host_id, neighbourhood_id, bathrooms_text, license.
static_feature_cols = [
    "listing_id",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "is_superhost",
    "host_listing_count",
    "neighbourhood_name",
]

static_features = base_listing_features[static_feature_cols].copy()

assert static_features["listing_id"].duplicated().sum() == 0

print(static_features.shape)
static_features.head()

## 10. Build review features in SQL

Do not load raw `core.review` into Pandas.

Build one row per listing in SQL.

Required output columns:

- `listing_id`
- `total_reviews_before_cutoff`
- `unique_reviewers_before_cutoff`
- `avg_comment_len_before_cutoff`
- `max_comment_len_before_cutoff`
- `days_since_last_review`

Use only reviews where `review_date <= cutoff_date`.

In [ ]:
review_features = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*) AS total_reviews_before_cutoff,
        COUNT(DISTINCT reviewer_pseudo_id) AS unique_reviewers_before_cutoff,
        AVG(comment_len) AS avg_comment_len_before_cutoff,
        MAX(comment_len) AS max_comment_len_before_cutoff,
        CAST(:cutoff_date AS date) - MAX(review_date) AS days_since_last_review
    FROM core.review
    WHERE review_date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={"cutoff_date": cutoff_date},
)

# Convert feature columns to numeric where needed.
numeric_review_cols = [
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
    "avg_comment_len_before_cutoff",
    "max_comment_len_before_cutoff",
    "days_since_last_review",
]

for col in numeric_review_cols:
    review_features[col] = pd.to_numeric(review_features[col], errors="coerce")

assert review_features["listing_id"].duplicated().sum() == 0

print(review_features.shape)
review_features.head()

## 11. Build calendar history features in SQL

Do not load raw `core.calendar_day` into Pandas.

Build historical availability features using:

- 90-day history window,
- 30-day recent history window.

Do not include calendar price features unless your audit proves they are usable.

In [ ]:
calendar_features_all = read_sql(
    """
    SELECT
        listing_id,
        -- 90-day window features
        COUNT(*) FILTER (
            WHERE date >= CAST(:history_start_date AS date)
              AND date <= CAST(:cutoff_date AS date)
        ) AS calendar_days_observed_last_90d,
        COUNT(*) FILTER (
            WHERE date >= CAST(:history_start_date AS date)
              AND date <= CAST(:cutoff_date AS date)
              AND available = true
        ) AS available_days_last_90d,
        CASE
            WHEN COUNT(*) FILTER (
                WHERE date >= CAST(:history_start_date AS date)
                  AND date <= CAST(:cutoff_date AS date)
            ) > 0 THEN
                COUNT(*) FILTER (
                    WHERE date >= CAST(:history_start_date AS date)
                      AND date <= CAST(:cutoff_date AS date)
                      AND available = true
                )::float / COUNT(*) FILTER (
                    WHERE date >= CAST(:history_start_date AS date)
                      AND date <= CAST(:cutoff_date AS date)
                )
            ELSE 0
        END AS available_rate_last_90d,
        AVG(minimum_nights) FILTER (
            WHERE date >= CAST(:history_start_date AS date)
              AND date <= CAST(:cutoff_date AS date)
        ) AS avg_minimum_nights_calendar_last_90d,
        AVG(maximum_nights) FILTER (
            WHERE date >= CAST(:history_start_date AS date)
              AND date <= CAST(:cutoff_date AS date)
        ) AS avg_maximum_nights_calendar_last_90d,
        -- 30-day recent history window features
        COUNT(*) FILTER (
            WHERE date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
              AND date <= CAST(:cutoff_date AS date)
        ) AS calendar_days_observed_last_30d,
        COUNT(*) FILTER (
            WHERE date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
              AND date <= CAST(:cutoff_date AS date)
              AND available = true
        ) AS available_days_last_30d,
        CASE
            WHEN COUNT(*) FILTER (
                WHERE date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
                  AND date <= CAST(:cutoff_date AS date)
            ) > 0 THEN
                COUNT(*) FILTER (
                    WHERE date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
                      AND date <= CAST(:cutoff_date AS date)
                      AND available = true
                )::float / COUNT(*) FILTER (
                    WHERE date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
                      AND date <= CAST(:cutoff_date AS date)
                )
            ELSE 0
        END AS available_rate_last_30d,
        AVG(minimum_nights) FILTER (
            WHERE date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
              AND date <= CAST(:cutoff_date AS date)
        ) AS avg_minimum_nights_calendar_last_30d,
        AVG(maximum_nights) FILTER (
            WHERE date >= CAST(:cutoff_date AS date) - INTERVAL '30 days'
              AND date <= CAST(:cutoff_date AS date)
        ) AS avg_maximum_nights_calendar_last_30d
    FROM core.calendar_day
    WHERE date >= CAST(:history_start_date AS date)
      AND date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "history_start_date": history_start_date,
        "cutoff_date": cutoff_date,
    },
)

# Convert numeric columns.
numeric_calendar_cols = [
    "calendar_days_observed_last_90d",
    "available_days_last_90d",
    "available_rate_last_90d",
    "avg_minimum_nights_calendar_last_90d",
    "avg_maximum_nights_calendar_last_90d",
    "calendar_days_observed_last_30d",
    "available_days_last_30d",
    "available_rate_last_30d",
    "avg_minimum_nights_calendar_last_30d",
    "avg_maximum_nights_calendar_last_30d",
]

for col in numeric_calendar_cols:
    calendar_features_all[col] = pd.to_numeric(calendar_features_all[col], errors="coerce")

assert calendar_features_all["listing_id"].duplicated().sum() == 0

print(calendar_features_all.shape)
calendar_features_all.head()

## 12. Build the target label

The label is built from future calendar availability.

Positive class:

```text
high_demand_proxy = 1 if future_available_rate_30d <= 0.30
```

This is not confirmed booking demand. It is a low-availability proxy.

In [ ]:
label_df = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*) AS future_calendar_days_observed_30d,
        COUNT(*) FILTER (WHERE available = true) AS future_available_days_30d,
        CASE
            WHEN COUNT(*) > 0 THEN
                COUNT(*) FILTER (WHERE available = true)::float / COUNT(*)
            ELSE 0
        END AS future_available_rate_30d,
        CASE
            WHEN COUNT(*) > 0
                 AND COUNT(*) FILTER (WHERE available = true)::float / COUNT(*) <= :threshold THEN
                1
            ELSE 0
        END AS high_demand_proxy
    FROM core.calendar_day
    WHERE date > CAST(:cutoff_date AS date)
      AND date <= CAST(:label_end_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "cutoff_date": cutoff_date,
        "label_end_date": label_end_date,
        "threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    },
)

# Convert numeric columns and make high_demand_proxy integer.
label_df["future_calendar_days_observed_30d"] = pd.to_numeric(
    label_df["future_calendar_days_observed_30d"], errors="coerce"
)
label_df["future_available_days_30d"] = pd.to_numeric(
    label_df["future_available_days_30d"], errors="coerce"
)
label_df["future_available_rate_30d"] = pd.to_numeric(
    label_df["future_available_rate_30d"], errors="coerce"
)
label_df["high_demand_proxy"] = label_df["high_demand_proxy"].astype(int)

assert label_df["listing_id"].duplicated().sum() == 0

print(label_df.shape)
label_df.head()

In [ ]:
# Check label balance.

label_distribution = (
    label_df["high_demand_proxy"]
    .value_counts(dropna=False)
    .rename_axis("high_demand_proxy")
    .reset_index(name="count")
)

label_distribution["percentage"] = (
    label_distribution["count"] / label_distribution["count"].sum()
).round(4)

label_distribution

## 13. Join feature groups and label

Join all feature groups into one ML-ready table.

The final granularity must be:

```text
one row = one listing_id at one cutoff_date
```

Use an inner join with `label_df`, because rows without a target cannot be used for supervised learning.

In [ ]:
# Join static_features, review_features, calendar_features_all, and label_df.
# Use inner join with label_df to ensure every row has a target.

feature_df = (
    label_df
    .merge(static_features, on="listing_id", how="inner")
    .merge(review_features, on="listing_id", how="left")
    .merge(calendar_features_all, on="listing_id", how="left")
)

# Add cutoff_date and dataset_version columns.
feature_df["cutoff_date"] = cutoff_date
feature_df["dataset_version"] = DATASET_VERSION

# Fill missing review count features with zero (listings with no reviews before cutoff).
review_fill_zero_cols = [
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
    "avg_comment_len_before_cutoff",
    "max_comment_len_before_cutoff",
]

for col in review_fill_zero_cols:
    feature_df[col] = feature_df[col].fillna(0)

# Handle missing days_since_last_review for listings with no reviews.
# Use a large sentinel value (e.g., PAST_WINDOW_DAYS) to indicate no review.
feature_df["days_since_last_review"] = feature_df["days_since_last_review"].fillna(PAST_WINDOW_DAYS)

assert feature_df["listing_id"].duplicated().sum() == 0

print(feature_df.shape)
feature_df.head()

## 14. Drop unusable columns

Before saving, remove bad feature columns.

Drop columns that are:

- more than 95% missing,
- constant across all rows,

but protect target/audit columns.

In [ ]:
protected_columns = {
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
}

HIGH_MISSING_DROP_THRESHOLD = 0.95

# Compute missing rates.
missing_rates = feature_df.isna().mean()

# Find high-missing columns outside protected columns.
high_missing_columns = [
    col for col in feature_df.columns
    if col not in protected_columns and missing_rates[col] > HIGH_MISSING_DROP_THRESHOLD
]

# Find constant columns outside protected columns.
constant_columns = [
    col for col in feature_df.columns
    if col not in protected_columns and feature_df[col].nunique(dropna=False) <= 1
]

columns_to_drop = list(set(high_missing_columns + constant_columns))

print("Columns to drop:", columns_to_drop)

feature_df = feature_df.drop(columns=columns_to_drop)

print("New shape:", feature_df.shape)

## 15. Validate the final dataset

The validation step is mandatory.

Check:

1. no duplicate `listing_id + cutoff_date`,
2. target exists and is binary,
3. no missing target values,
4. no forbidden PII columns,
5. no future leakage columns in model inputs.

In [ ]:
duplicate_count = feature_df.duplicated(subset=["listing_id", "cutoff_date"]).sum()
missing_target_count = feature_df["high_demand_proxy"].isna().sum()
unique_target_values = sorted(feature_df["high_demand_proxy"].dropna().unique().tolist())

forbidden_columns = {
    "host_id",
    "host_pseudo_id",
    "reviewer_id",
    "reviewer_pseudo_id",
    "review_id",
    "license",
    "bathrooms_text",
}

present_forbidden_columns = sorted(forbidden_columns.intersection(feature_df.columns))

label_only_columns = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

model_input_columns = [
    col for col in feature_df.columns
    if col not in label_only_columns
    and col not in ["listing_id", "cutoff_date", "dataset_version"]
]

future_leakage_columns = [
    col for col in model_input_columns
    if col.startswith("future_")
]

# Assert validation rules.
assert duplicate_count == 0, f"Expected 0 duplicates, got {duplicate_count}"
assert missing_target_count == 0, f"Expected 0 missing targets, got {missing_target_count}"
assert unique_target_values == [0, 1], f"Expected [0, 1], got {unique_target_values}"
assert len(present_forbidden_columns) == 0, f"Forbidden columns present: {present_forbidden_columns}"
assert len(future_leakage_columns) == 0, f"Future leakage columns: {future_leakage_columns}"

print("duplicate_count:", duplicate_count)
print("missing_target_count:", missing_target_count)
print("unique_target_values:", unique_target_values)
print("present_forbidden_columns:", present_forbidden_columns)
print("future_leakage_columns:", future_leakage_columns)
print("model_input_column_count:", len(model_input_columns))
print()
print("All validation assertions passed.")

In [ ]:
missing_report = (
    feature_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_rate"]

calendar_coverage_summary = (
    feature_df[["future_calendar_days_observed_30d"]]
    .describe()
    .reset_index()
)

display(missing_report.head(30))
display(label_distribution)
display(calendar_coverage_summary)

## 16. Save versioned outputs

Save:

- feature dataset,
- metadata,
- validation report,
- PII audit.

The MLflow notebook must read this output instead of querying raw database tables again.

In [ ]:
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"
validation_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_validation_report.json"
pii_audit_path = FEATURE_DIR / f"pii_audit_{DATASET_VERSION}.csv"

feature_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

try:
    feature_df.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except ImportError:
    print("Parquet not saved because pyarrow/fastparquet is not installed.")
    print("Install pyarrow with: pip install pyarrow")

# Build metadata dictionary.
metadata = {
    "dataset_version": DATASET_VERSION,
    "entity_column": ENTITY_COLUMN,
    "cutoff_date": str(cutoff_date),
    "history_start_date": str(history_start_date),
    "label_end_date": str(label_end_date),
    "past_window_days": PAST_WINDOW_DAYS,
    "future_window_days": FUTURE_WINDOW_DAYS,
    "high_demand_available_rate_threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    "target_definition": "high_demand_proxy = 1 if future_available_rate_30d <= threshold, else 0",
    "source_tables": [
        "core.listing",
        "core.host",
        "core.neighbourhood",
        "core.review (aggregated in SQL)",
        "core.calendar_day (aggregated in SQL)",
    ],
    "exclusion_rules": {
        "pii_columns_removed": [
            "host_id",
            "host_pseudo_id",
            "review_id",
            "reviewer_id",
            "reviewer_pseudo_id",
            "license",
            "bathrooms_text",
        ],
        "review_and_calendar_aggregated_in_sql": True,
        "columns_dropped_high_missing_or_constant": columns_to_drop,
    },
    "feature_columns": model_input_columns,
    "label_columns": label_only_columns,
    "feature_count": len(model_input_columns),
    "row_count": len(feature_df),
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Build validation_report dictionary.
validation_report = {
    "duplicate_listing_cutoff_rows": int(duplicate_count),
    "missing_target_count": int(missing_target_count),
    "target_values": unique_target_values,
    "present_forbidden_columns": present_forbidden_columns,
    "future_leakage_columns_in_model_inputs": future_leakage_columns,
    "missing_report_top30": missing_report.head(30).to_dict(orient="records"),
    "label_distribution": label_distribution.to_dict(orient="records"),
    "calendar_coverage_summary": calendar_coverage_summary.to_dict(orient="records"),
}

with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)

pii_audit.to_csv(pii_audit_path, index=False)

print("Saved metadata:", metadata_path)
print("Saved validation report:", validation_path)
print("Saved PII audit:", pii_audit_path)

## 17. Final preview

Use this final cell to confirm the output shape and columns.

Before moving to Notebook 2, make sure:

- target column exists,
- model input columns do not include future columns,
- no forbidden PII columns are present,
- saved files exist in `data/features/`.

In [ ]:
print("Final shape:", feature_df.shape)

display(feature_df.head())

feature_df.info()